# 03 — Vision Redaction via AWS Bedrock
Send each page image to Claude 3.7 Sonnet on Bedrock.

PII/PHI is **replaced with realistic fictitious dummy values** (not blanked).
The same original value gets the **same replacement** across all pages of a document.

Outputs per page stored in `redacted_text/` as `{stem}_page_{n}.json`:
```json
{ "sanitized_text": "...", "mapping": [{"original_masked": ..., "replacement": ..., "type": ...}] }
```

In [ ]:
%pip install pymupdf boto3 Pillow reportlab python-dotenv --prefer-binary -q

In [ ]:
import json, base64, os, sys, time
from pathlib import Path
from collections import defaultdict

# Add project root to path so we can import from models/
sys.path.insert(0, str(Path(os.getcwd()).parent))

from utils import (get_logger, extract_json, validate_mapping,
                   fix_remaining_violations, enforce_replacements_in_text,
                   audit_sanitized_text, check_duplicate_replacements,
                   fix_duplicate_replacements)
from models import get_bedrock_client, resolve_model_id, get_audit_model_id
logger = get_logger("03_redact_via_bedrock")

BASE_DIR      = Path(os.getcwd()).parent
TEMP_FOLDER   = BASE_DIR / "temp_images"
TEXT_FOLDER   = BASE_DIR / "redacted_text"

# Choose model: change this to any key from config/models.json
# e.g. "sonnet 3.7", "haiku 4.5", "sonnet 4.5", "opus 4.6"
BEDROCK_MODEL = resolve_model_id()  # default from config/models.json
AUDIT_MODEL   = get_audit_model_id()
MAX_TOKENS    = 4096
RETRY_LIMIT   = 3
RETRY_DELAY   = 5

TEXT_FOLDER.mkdir(exist_ok=True)
client = get_bedrock_client()
logger.info("Bedrock client ready — model: %s", BEDROCK_MODEL)

In [ ]:
BASE_PROMPT = """You are a data-sanitization assistant. For each page image, transcribe all \
text and replace every PII/PHI value with a completely different fictitious value that shares \
NO words with the original.

  Correct:  "MARGARET HOLMES" → "DIANA CHEN"
  Wrong:    "MARGARET HOLMES" → "MARGARET CHEN"  ← shares a word, not allowed
  Wrong:    "MARGARET HOLMES" → "MARGARET HOLMES" ← echoed the original, not allowed

Redact and replace ONLY these PII/PHI categories:
- Full names in any format ("First Last", "Last, First", with titles/suffixes/initials), \
including names after role labels such as "Claimant:", "Patient:", "Provider:", "Insured:", \
"Applicant:", "Member:", "Beneficiary:", etc.
- Email addresses
- Phone and fax numbers
- Social Security Numbers (SSN) or national identifiers
- Dates of Birth (DOB only — not date of service, date of injury, or other dates)
- Medical record numbers (MRN, patient ID, chart number, etc.)
- Credit card details (card numbers, expiration dates, CVVs, cardholder names)
- FEIN / Federal Employer Identification Number (EIN, Tax ID, TIN, Employer ID, \
federal tax identification number, or any employer tax number)
- State-issued ID numbers (driver's license, DL number, state ID card, or any variation)
- Professional / government-issued license numbers (nursing license, medical license, \
DEA number, NPI number, professional certification number, license/permit number)
- Physical mailing addresses — home/residential addresses, street addresses, and any \
address labeled "Home address", "Residence", "Personal address", "Mailing address", or \
similar personal address labels. IMPORTANT: Always redact the COMPLETE address as a single \
unit (street, city, state, and zip together). Never split an address into separate mapping \
rows for individual components (e.g. do NOT create separate entries for just a state code \
or just a zip code). Replace the entire address with a complete realistic fictitious address \
(e.g. "742 Elm Ave, Springfield, OH 44123").

Leave unchanged:
- Medical diagnoses or conditions (ICD codes, clinical descriptions)
- Business, facility, hospital, or clinic addresses (these are not personal PII)
- Insurance, policy, group, or claim numbers
- Non-DOB dates (date of service, injury, admission/discharge, etc.)
- Bank account or routing numbers
- Employer names or job titles
- Facility, hospital, or clinic names
- Organization, company, church, or entity names (e.g. "Pacific Ridge Casualty Company", \
"Diocese of Brooklyn", "First National Bank", "Blue Cross Blue Shield"). \
These are NOT personal names even if they appear after labels like "Insured:" or "Provider:".

Rules:
1. Transcribe ALL text — headings, labels, table cells, footers, and all form elements.
2. For checkboxes and form fields: reproduce the checked/unchecked state faithfully. \
Use [X] for checked boxes and [ ] for unchecked boxes. If a form has radio buttons or \
checkmarks (✓, ☑), transcribe them as [X]; empty boxes (☐, ○) as [ ]. \
Preserve which option is selected — do NOT leave all boxes unchecked.
3. For multi-column layouts (e.g. a form with a left section and right section): \
transcribe each column or section completely before moving to the next. \
Use a clear separator like "--- LEFT COLUMN ---" and "--- RIGHT COLUMN ---" or \
"[Section: Left]" / "[Section: Right]" to mark where each column starts. \
Do NOT interleave lines from different columns — finish one column fully, then start the next.
4. Replace every PII/PHI value inline. The replacement must be a completely different \
invented value — never echo the original. Use the same replacement whenever the same \
original appears.
5. Pay special attention to names inside table cells, form fields, and after role labels \
(e.g. "Claimant:", "Patient:"). These are PII regardless of formatting or context. \
However, do NOT redact organization, company, church, legal entity, or facility names — \
only redact names of individual people. "Pacific Ridge Casualty" is a company, not a person. \
"Diocese of Brooklyn" is an organization, not a person.
6. Replacements must look realistic but must not correspond to real people.
7. In the mapping, write only a partially masked hint of the original \
(e.g. "J**** S****" for "John Smith" — use the correct number of asterisks \
matching each hidden character). Never the full original value. \
The "replacement" field must contain the NEW fictitious value, never the original.
8. Wrap each replacement value in @ delimiters in the transcribed text \
(e.g. @Alex Rivera@, @456-78-9012@). In the mapping JSON, write the replacement WITHOUT @.
9. Each distinct original must get a unique replacement — never reuse the \
same replacement for different people or values.
10. Use these exact type labels in the mapping: Name, Email, Phone, SSN, DOB, MRN, \
CreditCard, FEIN, License, Address.

<<PRIOR_MAPPING>>
Return ONLY valid JSON with no markdown fences:
{
  "sanitized_text": "Claimant: @Alex Rivera@\\nSSN: @456-78-9012@\\n...",
  "mapping": [
    {"original_masked": "J**** S****", "replacement": "Alex Rivera", "type": "Name"},
    {"original_masked": "***-**-6789", "replacement": "456-78-9012", "type": "SSN"}
  ]
}

Now transcribe and sanitize the following document page:"""

BEDROCK_IMAGE_LIMIT = 5 * 1024 * 1024  # 5 MB — Bedrock limit on base64 string length


def image_to_base64(image_path: Path) -> str:
    from PIL import Image
    import io

    raw = image_path.read_bytes()
    encoded = base64.standard_b64encode(raw)
    if len(encoded) <= BEDROCK_IMAGE_LIMIT:
        return encoded.decode()

    img = Image.open(image_path).convert("RGB")
    for quality in (85, 70, 55, 40):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=quality, optimize=True)
        data = buf.getvalue()
        encoded = base64.standard_b64encode(data)
        if len(encoded) <= BEDROCK_IMAGE_LIMIT:
            logger.debug("Compressed %s to %.1f MB (quality=%d)",
                         image_path.name, len(data) / 1024 / 1024, quality)
            return encoded.decode()

    img = img.resize((img.width // 2, img.height // 2), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=70)
    data = buf.getvalue()
    logger.warning("Scaled down %s to %.1f MB", image_path.name, len(data) / 1024 / 1024)
    return base64.standard_b64encode(data).decode()


def build_prompt(prior_mapping: list[dict]) -> str:
    if not prior_mapping:
        return BASE_PROMPT.replace("<<PRIOR_MAPPING>>\n", "")
    rows = "\n".join(
        f"  - {r['original_masked']} → {r['replacement']} ({r['type']})"
        for r in prior_mapping
    )
    section = (
        "Previously identified mappings from earlier pages — "
        "use these EXACT replacements (wrapped in @) for any matching values on this page. "
        "Do NOT reuse any of these replacement values for a different original:\n"
        + rows + "\n\n"
    )
    return BASE_PROMPT.replace("<<PRIOR_MAPPING>>", section.rstrip())


def redact_image(image_path: Path, prior_mapping: list[dict]) -> dict:
    logger.debug("Encoding image: %s", image_path.name)
    img_b64 = image_to_base64(image_path)
    media_type = "image/jpeg" if img_b64[:4] == "/9j/" else "image/png"
    prompt  = build_prompt(prior_mapping)

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": media_type, "data": img_b64}},
                {"type": "text", "text": prompt}
            ]
        }]
    }

    last_exc = None
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            logger.debug("Invoking Bedrock (attempt %d/%d)", attempt, RETRY_LIMIT)
            resp = client.invoke_model(
                modelId=BEDROCK_MODEL, body=json.dumps(body),
                contentType="application/json", accept="application/json"
            )
            raw = json.loads(resp["body"].read())["content"][0]["text"]
            logger.debug("Raw response (first 200 chars): %s", raw[:200].replace("\n", "\\n"))
            result = extract_json(raw)

            violations = validate_mapping(result)
            if violations:
                logger.warning(
                    "Mapping violations in %s: %d row(s) share words with originals — retrying",
                    image_path.name, len(violations)
                )
                try:
                    rows_text = "\n".join(
                        f"  - {r['original_masked']} → {r['replacement']} ({r['type']})"
                        for r in violations
                    )
                    retry_body = {
                        "anthropic_version": "bedrock-2023-05-31",
                        "max_tokens": 1024,
                        "messages": [{"role": "user", "content": [{"type": "text", "text": (
                            "These replacements share words with their originals and must be fixed. "
                            "For each row, produce a new replacement that shares NO words with the original. "
                            "Return ONLY valid JSON (no markdown fences):\n"
                            "{\"mapping\": [{\"original_masked\": \"...\", "
                            "\"replacement\": \"...\", \"type\": \"...\"}]}\n\n"
                            f"Rows to fix:\n{rows_text}"
                        )}]}]
                    }
                    resp2 = client.invoke_model(
                        modelId=BEDROCK_MODEL, body=json.dumps(retry_body),
                        contentType="application/json", accept="application/json"
                    )
                    raw2 = json.loads(resp2["body"].read())["content"][0]["text"]
                    fixed = extract_json(raw2)
                    fix_map = {(r["original_masked"], r["type"]): r["replacement"]
                               for r in fixed.get("mapping", [])}
                    for row in result["mapping"]:
                        key = (row["original_masked"], row["type"])
                        if key in fix_map:
                            old_val = row["replacement"]
                            new_val = fix_map[key]
                            result["sanitized_text"] = result["sanitized_text"].replace(old_val, new_val)
                            row["replacement"] = new_val
                    remaining = validate_mapping(result)
                    if remaining:
                        logger.warning(
                            "%d violation(s) remain after retry in %s — applying synthetic fix",
                            len(remaining), image_path.name
                        )
                        fix_remaining_violations(result, remaining, logger)
                except Exception as exc2:
                    logger.error("Targeted retry failed for %s: %s", image_path.name, exc2)
                    fix_remaining_violations(result, violations, logger)

            enforce_replacements_in_text(result, logger)

            dupes = check_duplicate_replacements(result)
            if dupes:
                logger.warning("Duplicate replacements in %s: %d row(s) — fixing",
                               image_path.name, len(dupes))
                fix_duplicate_replacements(result, dupes, logger)

            audit_sanitized_text(result, client, AUDIT_MODEL, logger)

            return result

        except json.JSONDecodeError as exc:
            logger.warning(
                "Page %s: non-JSON response (image-only/banner page). "
                "Using raw text, empty mapping. Preview: %s",
                image_path.name, exc.doc[:200].replace("\n", "\\n")
            )
            return {"sanitized_text": exc.doc.strip(), "mapping": []}

        except Exception as exc:
            last_exc = exc
            logger.warning("Attempt %d failed for %s: %s", attempt, image_path.name, exc)
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)

    logger.error("All %d attempts exhausted for %s", RETRY_LIMIT, image_path.name)
    raise last_exc


def merge_mapping(accumulated: list[dict], new_rows: list[dict]) -> list[dict]:
    seen = {(r["original_masked"], r["type"]) for r in accumulated}
    added = 0
    for row in new_rows:
        key = (row["original_masked"], row["type"])
        if key not in seen:
            accumulated.append(row)
            seen.add(key)
            added += 1
    if added:
        logger.debug("Mapping: +%d new entries (total %d)", added, len(accumulated))
    return accumulated


logger.info("Prompt + helper functions defined")

In [ ]:
# Group images by PDF stem
image_files = sorted(TEMP_FOLDER.glob("*.png"))
logger.info("Found %d page image(s) in temp_images/", len(image_files))

grouped: dict[str, list[Path]] = defaultdict(list)
for img in image_files:
    grouped[img.stem.rsplit("_page_", 1)[0]].append(img)

for stem in grouped:
    grouped[stem].sort(key=lambda p: int(p.stem.rsplit("_page_", 1)[-1]))
    logger.info("  %s: %d page(s)", stem, len(grouped[stem]))

In [ ]:
# Run redaction — skips pages that already have a cached .json file
for pdf_stem, page_images in grouped.items():
    file_start = time.time()
    logger.info("=== Processing PDF: %s (%d pages) ===", pdf_stem, len(page_images))
    accumulated_mapping: list[dict] = []

    for page_idx, img_path in enumerate(page_images, 1):
        page_start = time.time()
        cache = TEXT_FOLDER / f"{img_path.stem}.json"

        if cache.exists():
            logger.info("[cache] %s", img_path.name)
            data = json.loads(cache.read_text(encoding="utf-8"))
            violations = validate_mapping(data)
            if violations:
                logger.warning("[cache] %d violation(s) in cached %s — fixing",
                               len(violations), img_path.name)
                fix_remaining_violations(data, violations, logger)
            enforce_replacements_in_text(data, logger)
            audit_sanitized_text(data, client, AUDIT_MODEL, logger)
        else:
            logger.info("Redacting %s ...", img_path.name)
            data = redact_image(img_path, prior_mapping=accumulated_mapping)
            cache.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
            logger.info("Done: %s — %d mapping entries this page", img_path.name, len(data.get("mapping", [])))

        # Cross-page duplicate check: compare this page against all prior pages
        dupes = check_duplicate_replacements(data, prior_mapping=accumulated_mapping)
        if dupes:
            logger.warning("Cross-page duplicate replacements on page %d: %d row(s) — fixing",
                           page_idx, len(dupes))
            fix_duplicate_replacements(data, dupes, logger, prior_mapping=accumulated_mapping)
            cache.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

        accumulated_mapping = merge_mapping(accumulated_mapping, data.get("mapping", []))

        page_elapsed = time.time() - page_start
        logger.info("  Page %d/%d — %.1fs", page_idx, len(page_images), page_elapsed)

    file_elapsed = time.time() - file_start
    logger.info("PDF complete: %s — %d unique replacements, %.1fs total (%.1fs/page avg)",
                pdf_stem, len(accumulated_mapping), file_elapsed,
                file_elapsed / len(page_images))

logger.info("Redaction finished for all PDFs")

In [ ]:
# Preview: first page sanitized text + mapping table for first PDF
json_files = sorted(TEXT_FOLDER.glob("*.json"))
if not json_files:
    logger.warning("No output files found")
else:
    first = json.loads(json_files[0].read_text(encoding="utf-8"))
    logger.info("Preview: %s", json_files[0].name)
    print(f"\n=== {json_files[0].name} — sanitized text ===")
    print(first["sanitized_text"])

    first_stem = json_files[0].stem.rsplit("_page_", 1)[0]
    all_rows: list[dict] = []
    for jf in sorted(TEXT_FOLDER.glob(f"{first_stem}_page_*.json"),
                     key=lambda p: int(p.stem.rsplit("_page_", 1)[-1])):
        data = json.loads(jf.read_text(encoding="utf-8"))
        for row in data.get("mapping", []):
            key = (row["original_masked"], row["type"])
            if key not in {(r["original_masked"], r["type"]) for r in all_rows}:
                all_rows.append(row)

    print(f"\n=== Summary table — {first_stem} ({len(all_rows)} entries) ===")
    print(f"{'Original (masked)':<28} {'Replacement':<28} {'Type'}")
    print("-" * 70)
    for row in all_rows:
        print(f"{row['original_masked']:<28} {row['replacement']:<28} {row['type']}")